# Student At-Risk Classification – Solution

**Short name (GitHub):** `EduRisk`  
**Lab source:** MushEdib pipeline adapted to an education / early-warning codebook.  
**Data:** `data/students.csv` (7,208 × 24 letter codes, synthetic SIS extract). Target `status`: **o** = on-track, **r** = at-risk.  
**Companion files:** `EduRisk_Solution.ipynb`, `EduRisk_Reusable_Template.ipynb`, `EduRisk.py`, `EduRisk_Cheatsheet.docx`, `EduRisk_Project_Memo.docx`, `EduRisk_Strategy_Guide.docx`, `EduRisk_1Page_Summary_Report.docx`, `edurisk_flowchart.png`.

This notebook is the worked key. Use `EduRisk_Practice_Skeleton.ipynb` to practice first.

**This is not a grading, placement, scholarship, or expulsion engine.** A hold-out score on this codebook does not transfer to a live roster.

You will:

1. Clean `?` in `advisor` → `u`, drop 8 exact duplicates.
2. Plot status balance, attendance × status, prior-gpa × status, and a 12-feature factorize heatmap.
3. Drop zero-variance `roster-flag`, LabelEncode every column, 80/20 split (`random_state=42`).
4. Fit `RandomForestClassifier(random_state=42)` and evaluate.
5. Compare alternates, run extra practice, twist simulation knobs.



## Inline cheat-sheet (keep this cell visible)

See also **`EduRisk_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Load | `pd.read_csv("data/students.csv")` |
| Missing in this table | literal `"?"` in `advisor` (~2,073 rows) |
| Task replacement | `df["advisor"] = df["advisor"].replace("?", "u")` |
| Collision | `u` here means *unknown advisor contact*, not a grade |
| Duplicates | `df.duplicated().sum()` then `drop_duplicates()` — expect 8 |
| Zero-variance | `roster-flag` is always `e` (enrolled) — drop before encoding |
| Factorize (EDA only) | `df[col], _ = pd.factorize(df[col])` — arbitrary integer codes |
| Top-12 heatmap | drop `status` *before* `.head(12)`; heatmap is 12×12, not 13×13 |
| Encode for trees | `LabelEncoder().fit_transform` per column |
| Encode for distance / LR | `pd.get_dummies` (one-hot) |
| Split | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| RF | `RandomForestClassifier(random_state=42)` — defaults, 100 trees |
| Costly cell | actual **r**, predicted **o** (missed at-risk / skipped intervention) |
| Attendance rule | majority status per attendance code ≈ 0.71 on this table |
| Class map after LE | alphabetical → `o=0`, `r=1` |

**sklearn note.** Trees do not need scaling. LabelEncoder invents a fake order that is fine for RF / DT and wrong for kNN / unpenalized linear models. Factorize |corr| can rank a weak column above attendance — trust Gini / MI for the key feature.



## Flowchart of the desired outcome

![EduRisk flow](edurisk_flowchart.png)

Clean first (unknown advisor, drop the 8 cloned rows). Look at attendance before you fit anything. Drop `roster-flag`. Freeze the 80/20 seed at 42. Score the model on the costly cell (missed at-risk), not only accuracy. Then break it on purpose in the simulation section.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    recall_score, f1_score,
)
from sklearn.feature_selection import mutual_info_classif

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

try:
    import EduRisk as er
except ImportError:
    er = None
print("ready")


## 1. Data preparation

Load `data/students.csv`. Display the first 5 rows. Call `.info()`. Count `"?"` in `advisor` and replace them with `"u"`. Count and drop exact duplicate rows. Print the cleaned shape and `status` counts.

Expected: 7,208 raw rows × 24 columns, ~2,073 question marks, 8 duplicates → **7,200** cleaned rows. Status split about 3,832 on-track / 3,368 at-risk.


In [ ]:
df = pd.read_csv("data/students.csv")
print("First 5 rows:")
print(df.head())
print("\nDataFrame info:")
df.info()
n_q = int((df["advisor"] == "?").sum())
print(f"\nMissing ('?') values in 'advisor': {n_q}")
df["advisor"] = df["advisor"].replace("?", "u")
assert (df["advisor"] == "?").sum() == 0
n_dups = int(df.duplicated().sum())
print(f"Number of duplicate rows: {n_dups}")
if n_dups:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed.")
print(f"Cleaned dataset shape: {df.shape}")
print(df["status"].value_counts())


### Alternate — treat `?` as `missing`, or impute the mode

The brief asks for `"u"`. Two other legal choices: a new token `"missing"`, or the mode of observed advisor codes (`b` = brief). Trees can learn from an explicit missing level; imputing the mode invents a contact pattern that was never recorded.


In [ ]:
raw = pd.read_csv("data/students.csv")
obs_mode = raw.loc[raw["advisor"] != "?", "advisor"].mode().iloc[0]
print("mode of raw advisor (excluding ?):", obs_mode)
df_missing = raw.copy()
df_missing["advisor"] = df_missing["advisor"].replace("?", "missing")
df_mode = raw.copy()
df_mode["advisor"] = df_mode["advisor"].replace("?", obs_mode)
print("unknown-as-missing levels:", sorted(df_missing["advisor"].unique()))
print("mode-imputed levels:", sorted(df_mode["advisor"].unique()))
print("We keep df with '?' → 'u' for the rest of the notebook.")


## 2. Exploratory data analysis

Plot three countplots, then a temporary factorize encoding, then the **12 × 12** heatmap of the top features by |corr| with `status`.

1. Status balance.
2. Attendance vs status (the key feature).
3. Prior-GPA vs status (overlap — weaker on its own).
4. `pd.factorize` every column into integers.
5. Absolute correlation with `status`, drop the target, take `.head(12)`, heatmap **those 12 only**.

Reference images: `edurisk_class_balance.png`, `edurisk_attendance.png`, `edurisk_priorgpa.png`, `edurisk_heatmap.png`.

**Caveat.** Factorize assigns arbitrary integers, so |corr| can put `homework` above `attendance` even when Gini / MI agree attendance is the driver. Treat the heatmap as a sketch.


In [ ]:
fig, ax = plt.subplots()
ax = sns.countplot(data=df, x="status", hue="status",
                   palette={"o": "#6aa84f", "r": "#cc4125"}, legend=False, ax=ax)
ax.set_title("Status balance: on-track vs at-risk")
ax.set_xlabel("status (o = on-track, r = at-risk)")
total = len(df)
for container in ax.containers:
    labels = [f"{int(v.get_height())}\n({v.get_height()/total:.1%})" for v in container]
    ax.bar_label(container, labels=labels, padding=3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=df, x="attendance", hue="status",
              palette={"o": "#6aa84f", "r": "#cc4125"}, ax=ax)
ax.set_title("At-risk status by attendance (key feature)")
ax.set_xlabel("attendance  a=always u=usually s=sometimes r=rarely n=never")
plt.tight_layout(); plt.show()

order = df["prior-gpa"].value_counts().index
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="prior-gpa", hue="status", order=order,
              palette={"o": "#6aa84f", "r": "#cc4125"}, ax=ax)
ax.set_title("At-risk status by prior GPA band")
ax.set_xlabel("prior-gpa  h=high m=mid l=low")
plt.tight_layout(); plt.show()

df_encoded = df.copy()
for col in df_encoded.columns:
    df_encoded[col], _ = pd.factorize(df_encoded[col])
print("Encoded preview:")
print(df_encoded.head())

correlations = df_encoded.corr()["status"].abs().drop("status")
top12 = correlations.sort_values(ascending=False).head(12)
print("\nTop 12 features correlated with status:")
print(top12)

top_features = list(top12.index)  # exactly 12 — do not append status
corr_matrix = df_encoded[top_features].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="Purples",
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation heatmap of top 12 features")
plt.tight_layout(); plt.show()


### What you should see

- Balance is close: **~53% on-track / ~47% at-risk**. Accuracy is usable; the *costly* error is still a missed at-risk flag.
- Attendance nearly partitions the tails: `a` (always) is mostly on-track; `r` / `n` (rarely / never) are mostly at-risk. `s` and `u` overlap — that is where homework, GPA, and tardies earn their keep.
- Prior GPA overlaps both classes. High GPA helps but does not guarantee on-track.
- Expect factorize |corr| to mention homework, prior-gpa, attendance, extra-help. Confirm with Gini / MI later.


## 3. Preprocessing

Drop `roster-flag`. LabelEncode **every remaining column including `status`**. Split `X` / `y`. `train_test_split(..., test_size=0.2, random_state=42)`. Print the four shapes.

Expected shapes: `X_train (5760, 22)`, `X_test (1440, 22)`.


In [ ]:
df_model = df.drop(columns=["roster-flag"])
print(f"Dropped 'roster-flag'. Remaining columns: {len(df_model.columns)}")

label_encoders = {}
df_le = df_model.copy()
for col in df_le.columns:
    le = LabelEncoder()
    df_le[col] = le.fit_transform(df_le[col])
    label_encoders[col] = le
print("Encoded target classes:", dict(zip(
    label_encoders["status"].classes_,
    label_encoders["status"].transform(label_encoders["status"].classes_),
)))

X = df_le.drop(columns=["status"])
y = df_le["status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("test at-risk rate:", float(y_test.mean()))


### Alternate — one-hot instead of LabelEncoder

Keep this for logistic regression later. Do not replace `X_train` used by the forest.


In [ ]:
X_oh = pd.get_dummies(df.drop(columns=["roster-flag", "status"]), drop_first=False)
y_oh = (df["status"] == "r").astype(int)
print("one-hot width:", X_oh.shape[1])
Xoh_train, Xoh_test, yoh_train, yoh_test = train_test_split(
    X_oh, y_oh, test_size=0.2, random_state=42,
)


## 4. Random Forest

Initialize `RandomForestClassifier(random_state=42)`, fit on the training fold, predict `X_test` into `y_pred`. Name the model `clf` so the evaluation cell matches the brief.


In [ ]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("predicted at-risk (r=1) count:", int(y_pred.sum()))


## 5. Model evaluation

Print accuracy, the numeric confusion matrix, and the classification report. Heatmap the matrix. Horizontal bar of the **top 5** Gini importances.

On this seed the default forest sits near **0.76** accuracy. Watch the costly cell: actual at-risk, predicted on-track (missed intervention).


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy score:", accuracy)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)
print("Classification report:")
print(classification_report(
    y_test, y_pred, target_names=["on-track (o=0)", "at-risk (r=1)"],
))
print("missed at-risk (costly cell):", int(((y_test == 1) & (y_pred == 0)).sum()))

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", square=True,
            xticklabels=["on-track", "at-risk"],
            yticklabels=["on-track", "at-risk"],
            cbar_kws={"shrink": 0.8})
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion matrix — Random Forest")
plt.tight_layout(); plt.show()

importances = pd.Series(clf.feature_importances_, index=X_train.columns)
top5 = importances.sort_values(ascending=False).head(5)
print(importances.sort_values(ascending=False).head(8))
plt.figure(figsize=(8, 5))
ax = sns.barplot(x=top5.values, y=top5.index, orient="h", color="#6d4aff")
ax.set_title("Top 5 feature importances — Random Forest")
ax.set_xlabel("Gini importance")
for i, v in enumerate(top5.values):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center")
plt.tight_layout(); plt.show()


## 6. Alternate code that reaches a similar decision

Fit a single `DecisionTreeClassifier`, a one-hot `LogisticRegression`, an attendance-majority rule, and a mutual-information ranking. On this table the one-hot logistic model is often a hair *above* the forest; the attendance rule sits near **0.71**.


In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
print("DT acc", accuracy_score(y_test, dt.predict(X_test)))

lr = LogisticRegression(max_iter=2000, solver="liblinear")
lr.fit(Xoh_train, yoh_train)
print("one-hot LogReg acc", accuracy_score(yoh_test, lr.predict(Xoh_test)),
      "width", X_oh.shape[1])

maj = df.groupby("attendance")["status"].agg(lambda s: s.value_counts().idxmax())
att_pred = df["attendance"].map(maj)
att_acc = float((att_pred == df["status"]).mean())
print("attendance majority-rule acc (full table)", att_acc)
print("attendance mix:\n", pd.crosstab(df["attendance"], df["status"]))

mi = pd.Series(
    mutual_info_classif(X, y, random_state=42), index=X.columns,
).sort_values(ascending=False)
print("\nMutual information with status:")
print(mi.head(8))


## 7. More practice

**A.** Restrict the test fold to `setting == r` (rural). Does accuracy hold?

**B.** Cost matrix. Treat a missed at-risk flag as 5× worse than a false alarm. Sweep `predict_proba` thresholds that flag at-risk more aggressively. How many extra false alarms buy a drop in missed-risk cells?

**C.** A 2-feature card: `attendance` + `homework` only. Compare test accuracy and the costly-cell count to the full 22-feature forest.


In [ ]:
# A — rural setting on the encoded frame
rural_code = label_encoders["setting"].transform(["r"])[0]
rural = X_test["setting"] == rural_code
print("rural test rows:", int(rural.sum()),
      "acc", accuracy_score(y_test[rural], y_pred[rural]) if rural.any() else None)

# B — probability threshold
proba = clf.predict_proba(X_test)[:, 1]  # P(at-risk)
print("\nthreshold | missed_risk | false_alarm | acc | at-risk recall")
for t in [0.50, 0.40, 0.30, 0.20]:
    pred_t = (proba >= t).astype(int)
    missed = int(((y_test == 1) & (pred_t == 0)).sum())
    alarm = int(((y_test == 0) & (pred_t == 1)).sum())
    acc_t = accuracy_score(y_test, pred_t)
    rec_t = recall_score(y_test, pred_t, pos_label=1)
    print(f"  {t:4.2f}     | {missed:11d} | {alarm:11d} | {acc_t:.3f} | {rec_t:.3f}")

# C — attendance + homework
cols2 = ["attendance", "homework"]
clf2 = RandomForestClassifier(random_state=42)
clf2.fit(X_train[cols2], y_train)
yp2 = clf2.predict(X_test[cols2])
print("\n2-feature acc", accuracy_score(y_test, yp2),
      "missed at-risk", int(((y_test == 1) & (yp2 == 0)).sum()))


## 8. Simulation — twist a few knobs

Default RF is *not* perfect here (unlike MushEdib). Knobs that move:

| Knob | What we change | What usually happens on this table |
|------|----------------|------------------------------------|
| `max_depth` | 1 → None | depth 1 ≈ 0.65; depth 8 ≈ 0.75 |
| drop features | remove attendance / homework | drops toward 0.61–0.65; attendance-only ≈ 0.71 |
| label flip | flip 0–35% of *train* labels | holds near 0.75 until ~10%, then falls |
| training n | 50, 100, …, 5,760 | 50 rows ≈ 0.59; 800 rows already ~0.75 |

Edit the lists, re-run, read the 2×2 panel. Reference: `edurisk_simulation.png`.


In [ ]:
rng = np.random.default_rng(42)
DEPTHS = [1, 2, 3, 4, 5, 6, 8, None]
FLIP_RATES = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
TRAIN_NS = [50, 100, 200, 400, 800, 1600, 3200, len(X_train)]

fig, axes = plt.subplots(2, 2, figsize=(10.6, 7.8))

dacc = []
for d in DEPTHS:
    m = RandomForestClassifier(n_estimators=50, max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    dacc.append(accuracy_score(y_test, m.predict(X_test)))
labs = ["None" if d is None else str(d) for d in DEPTHS]
axes[0, 0].plot(range(len(DEPTHS)), dacc, marker="o", color="#cc4125")
axes[0, 0].set_xticks(range(len(DEPTHS))); axes[0, 0].set_xticklabels(labs)
axes[0, 0].set_title("Accuracy vs max_depth (50 trees)")
axes[0, 0].set_xlabel("max_depth"); axes[0, 0].set_ylabel("test acc")
print("depth", list(zip(labs, [round(a, 4) for a in dacc])))

drop_plan = {
    "all 22": [],
    "no attendance": ["attendance"],
    "no attendance+hw": ["attendance", "homework"],
    "no attendance+gpa": ["attendance", "prior-gpa"],
    "only attendance": None,
}
names, accs = [], []
for name, cols in drop_plan.items():
    if name == "only attendance":
        Xt, Xe = X_train[["attendance"]], X_test[["attendance"]]
    else:
        Xt, Xe = X_train.drop(columns=cols), X_test.drop(columns=cols)
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(Xt, y_train)
    names.append(name); accs.append(accuracy_score(y_test, m.predict(Xe)))
axes[0, 1].barh(names, accs, color="#6d4aff")
axes[0, 1].set_xlim(0.55, 1.0)
axes[0, 1].set_title("Accuracy after dropping key features")
print("drop", list(zip(names, [round(a, 4) for a in accs])))

nacc, nrec = [], []
ytr_np = y_train.to_numpy()
for f in FLIP_RATES:
    y_noisy = ytr_np.copy()
    k = int(f * len(y_noisy))
    if k:
        idx = rng.choice(len(y_noisy), size=k, replace=False)
        y_noisy[idx] = 1 - y_noisy[idx]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(X_train, y_noisy)
    yp = m.predict(X_test)
    nacc.append(accuracy_score(y_test, yp))
    nrec.append(recall_score(y_test, yp, pos_label=1))
axes[1, 0].plot([f * 100 for f in FLIP_RATES], nacc, marker="o", label="accuracy", color="#6aa84f")
axes[1, 0].plot([f * 100 for f in FLIP_RATES], nrec, marker="s", label="at-risk recall", color="#cc4125")
axes[1, 0].set_title("Train label-flip vs test metrics")
axes[1, 0].set_xlabel("flipped labels (%)"); axes[1, 0].legend()
print("noise acc", list(zip(FLIP_RATES, [round(a, 4) for a in nacc])))

sacc = []
perm = rng.permutation(len(X_train))
for n in TRAIN_NS:
    take = perm[:n]
    m = RandomForestClassifier(n_estimators=50, random_state=42)
    m.fit(X_train.iloc[take], y_train.iloc[take])
    sacc.append(accuracy_score(y_test, m.predict(X_test)))
axes[1, 1].plot(TRAIN_NS, sacc, marker="o", color="#e69138")
axes[1, 1].set_title("Accuracy vs random training n")
axes[1, 1].set_xlabel("training rows")
print("n-sub", list(zip(TRAIN_NS, [round(a, 4) for a in sacc])))

plt.suptitle("EduRisk simulation knobs")
plt.tight_layout(); plt.show()


## 9. Audience notes (rewrite the same result four ways)

Use the attached audience PDFs. Same numbers, four pitches.

| Audience | Data literacy | Subject knowledge | Time span | What to show |
|----------|---------------|-------------------|-----------|--------------|
| Expert (education researcher) | high | high | long | MI vs Gini vs factorize, why 0.76 is not a placement rule, `u` vs mode impute |
| Technician (SIS / early-warning) | medium | high practical | short | 2-feature card, threshold as intervention capacity, do not write to the gradebook |
| Executive (dean / superintendent) | low–medium | low–medium | very short | 53 / 47 balance, ~0.76 acc, 183 missed-risk on hold-out, human review required |
| Nonspecialist (family) | low | low | short | “showing up is the loud clue in *this* table, not a verdict about your child” |

Full prose: `EduRisk_Project_Memo.docx`.



## 10. Good fit vs limitations

**Good fit**
- All-categorical SIS / early-warning codebook with a nearly balanced binary target.
- Tree ensembles and a one-hot logistic baseline.
- Teaching clean vs mode-impute, LabelEncoder vs one-hot, costly-error thinking (missed intervention).

**Limitations / anti-applications**
- Synthetic extract. New schools, different coding manuals, and mid-term grade streams are out of scope.
- `?` → `u` is a teaching token, not a district standard.
- ~0.76 hold-out accuracy is *not* a reason to auto-assign tutoring, hold a student back, or deny a scholarship.
- Never a grading engine, never an expulsion score, never a public “will my child fail?” app.

Top applications of the *pattern* (categorical RF + costly FN): attendance early-warning, library-fines typology, course-add codes — always with a counselor in the loop.

